In [5]:
import pandas as pd
import import_ipynb
import nbimporter
from utils import prepareTrainingSet3Sec,prepareTrainingSet30Sec,standardization,metrics,Grafico_Tre_Valori, ConfrontoGrafico_30_e_3, evaluate_Model, prepareTrainingSet30Sec_Arg , prepareTrainingSet3Sec_Arg  #type: ignore
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV  
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score

In [23]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Prepara i dati
X_train, X_test, y_train, y_test = prepareTrainingSet3Sec()

# Converti le etichette delle classi in valori numerici
label_encoder = LabelEncoder()
y_test = label_encoder.fit_transform(y_test)
y_train = label_encoder.fit_transform(y_train)

# utilizzo la funzione di standardizzazzione presente in utils che usa Standard Scaler
X_train, X_test = standardization(X_train, X_test)

# Creare il modello Logistic
model = XGBClassifier(learning_rate=0.01, n_estimators=600, max_depth=9, min_child_weight=25, subsample=0.8, colsample_bytree=0.8)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
evaluate_Model(y_test, y_pred, y_train, y_pred_train)

---------STATISTICHE TEST----------
Accuratezza: 0.8509
Precision: 0.8497
Recall: 0.8513
F1-score: 0.8495
F2-score: 0.8503
---------STATISTICHE TRAING----------
Accuratezza: 0.9439
Precision: 0.9443
Recall: 0.9439
F1-score: 0.9439
F2-score: 0.9439

=== Analisi ===
Il modello sembra bilanciato e generalizza bene.


In [4]:
import xgboost, sklearn
print(xgboost.__version__)
print(sklearn.__version__)

2.1.2
1.6.1


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Caricare il dataset
DATASET_CSV = "Data/features_3_sec.csv"
df = pd.read_csv(DATASET_CSV)

# Separare features e target
feature_cols = [col for col in df.columns if col not in ['filename', 'label']]
X = df[feature_cols].values.astype(float)
y = df['label'].values

# Suddividere in training e test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

label_encoder = LabelEncoder()
y_test = label_encoder.fit_transform(y_test)
y_train = label_encoder.fit_transform(y_train)

# Definire la griglia degli iperparametri
param_grid = {
    'n_estimators': [100, 200],  # Numero di alberi
    'max_depth': [3, 5, 7],  # Profondità massima degli alberi
    'learning_rate': [ 0.1, 0.3],  # Tasso di apprendimento
    
}

# Creare il modello
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

# Eseguire la Grid Search con cross-validation
grid_search = GridSearchCV(model, param_grid, scoring='accuracy', cv=5, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Stampare i migliori parametri
print("Migliori parametri trovati:", grid_search.best_params_)

# Valutare il miglior modello
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

valid_models = []
for params in grid_search.cv_results_['params']:
    model.set_params(**params)
    model.fit(X_train, y_train)
    train_accuracy = accuracy_score(y_train, model.predict(X_train))
    test_accuracy = accuracy_score(y_test, model.predict(X_test))
    if abs(train_accuracy - test_accuracy) <= train_accuracy*0.1 and train_accuracy < 0.99:
        valid_models.append((model, params, train_accuracy, test_accuracy))

# Trova il miglior modello tra quelli validi
if valid_models:
    best_model, best_params, best_train_accuracy, best_test_accuracy = max(valid_models, key=lambda x: x[3])
    randomForest=0.7 #{best_test_accuracy}
    print(f"Training Accuracy: {best_train_accuracy}")
    print(f"Testing Accuracy: {best_test_accuracy}")
    print(f"Best Parameters: {best_params}")
    print("La differenza tra l'accuracy di training e di test è entro il 10%.")
else:
    print("Nessun modello ha una differenza di accuracy entro il 10%.")

C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [21]:
def standardization2(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    return X_train, X_val, X_test


In [25]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Caricare il dataset
DATASET_CSV = "Data/features_3_sec.csv"
df = pd.read_csv(DATASET_CSV)

# Separare features e target
feature_cols = [col for col in df.columns if col not in ['filename', 'label']]
X = df[feature_cols].values.astype(float)
y = df['label'].values

# Suddividere in Training (70%), Validation (15%), Test (15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Convertire le etichette in numeri interi
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

# Standardizzazione dei dati
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Creare il modello XGBoost con parametri ottimizzati
model = XGBClassifier(
    learning_rate=0.1,  
    n_estimators=150,   
    max_depth=5,        
    eval_metric='mlogloss'
)

# Allenare il modello su training set e validazione
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=10, verbose=True)

# Predizioni su training, validation e test set
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

# Valutazione su training, validation e test set
def evaluate_model(y_true, y_pred, dataset_name):
    acc = accuracy_score(y_true, y_pred)
    print(f"\n--------- STATISTICHE {dataset_name.upper()} ---------")
    print(f"Accuratezza: {acc:.4f}")
    print(classification_report(y_true, y_pred))

# Stampare le metriche per verificare l'overfitting
evaluate_model(y_train, y_pred_train, "Training")
evaluate_model(y_val, y_pred_val, "Validation")
evaluate_model(y_test, y_pred_test, "Test")


[0]	validation_0-mlogloss:2.11216
[1]	validation_0-mlogloss:1.97495


C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[2]	validation_0-mlogloss:1.86152
[3]	validation_0-mlogloss:1.76735
[4]	validation_0-mlogloss:1.68564
[5]	validation_0-mlogloss:1.61198
[6]	validation_0-mlogloss:1.54860
[7]	validation_0-mlogloss:1.49005
[8]	validation_0-mlogloss:1.43647
[9]	validation_0-mlogloss:1.38700
[10]	validation_0-mlogloss:1.34238
[11]	validation_0-mlogloss:1.29785
[12]	validation_0-mlogloss:1.26055
[13]	validation_0-mlogloss:1.22557
[14]	validation_0-mlogloss:1.19182
[15]	validation_0-mlogloss:1.16092
[16]	validation_0-mlogloss:1.13106
[17]	validation_0-mlogloss:1.10348
[18]	validation_0-mlogloss:1.07718
[19]	validation_0-mlogloss:1.05293
[20]	validation_0-mlogloss:1.03125
[21]	validation_0-mlogloss:1.00994
[22]	validation_0-mlogloss:0.99085
[23]	validation_0-mlogloss:0.97395
[24]	validation_0-mlogloss:0.95512
[25]	validation_0-mlogloss:0.93755
[26]	validation_0-mlogloss:0.92033
[27]	validation_0-mlogloss:0.90444
[28]	validation_0-mlogloss:0.88921
[29]	validation_0-mlogloss:0.87540
[30]	validation_0-mlogloss:0

KeyboardInterrupt: 